# Role 3 — multi-sample ReVA validation and inference benchmark

This Colab notebook runs the committed Role 3 video-to-temporal pipeline on a deterministic subset of **32 actual validation QA records**, sampled round-robin across observed categories and preferring distinct videos. It reports feature extraction only; it does **not** evaluate answer accuracy or claim temporal reasoning improvement.

Select **Runtime → Change runtime type → T4 GPU** when available. The notebook clones `suraj-role3-foundation` and `sarthak` into separate runtime directories. The validation annotation revision is pinned from Sarthak's manifest. Only selected video files are fetched lazily.

For each of T=4, 8, and 16, it records per-sample success/failure and feature shapes. It then benchmarks one same validation sample and batch size 1 using 3 warm-ups and 10 measured forward passes per frame count. Each timed GPU pass is synchronized before and after. Video fetch/decode, sampling, Sarthak resize/normalization, and host-to-device transfer are outside the timer; extractor CLIP normalization, CLIP forward, and temporal Transformer forward are inside.

Missing/corrupt videos and CUDA OOMs are recorded. No placeholder frames or synthetic features are substituted.

In [ ]:
%pip install -q "transformers>=4.40,<5" "huggingface_hub>=0.23,<1" opencv-python-headless

# Colab supplies PyTorch and NumPy; they are not reinstalled.

In [ ]:
import torch
import platform

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = None
GPU_TOTAL_MEMORY_MB = None
GPU_FREE_MEMORY_MB = None
if torch.cuda.is_available():
    GPU_NAME = torch.cuda.get_device_name(0)
    free_bytes, total_bytes = torch.cuda.mem_get_info(0)
    GPU_TOTAL_MEMORY_MB = round(total_bytes / 1024**2, 2)
    GPU_FREE_MEMORY_MB = round(free_bytes / 1024**2, 2)
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
print("GPU:", GPU_NAME)
print("GPU memory MiB free/total:", GPU_FREE_MEMORY_MB, GPU_TOTAL_MEMORY_MB)

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/sarthak-shastri/ChronoMind.git"
ROLE3_BRANCH = "suraj-role3-foundation"
PREPROCESSING_BRANCH = "sarthak"
ROLE3_ROOT = Path("/content/ChronoMind-role3")
SARTHAK_ROOT = Path("/content/ChronoMind-sarthak")
EXPECTED_ROLE3_COMMIT = "62f383676b63717702ae27eb52e023a74c2e1d38"
EXPECTED_PREPROCESSING_COMMIT = "1a63b5009267a348b5fdd550b063869332d4ce0a"

def clone_or_verify(branch, destination):
    if not destination.exists():
        subprocess.run(
            ["git", "clone", "--quiet", "--single-branch", "--branch", branch,
             REPO_URL, str(destination)],
            check=True, text=True, capture_output=True,
        )
        return
    current = subprocess.run(
        ["git", "-C", str(destination), "branch", "--show-current"],
        check=True, text=True, capture_output=True,
    ).stdout.strip()
    dirty = subprocess.run(
        ["git", "-C", str(destination), "status", "--porcelain"],
        check=True, text=True, capture_output=True,
    ).stdout.strip()
    if current != branch or dirty:
        raise RuntimeError(
            f"Existing runtime checkout {destination} is on {current!r} or dirty. "
            "Use a fresh Colab runtime; this notebook will not switch or overwrite it."
        )

clone_or_verify(ROLE3_BRANCH, ROLE3_ROOT)
clone_or_verify(PREPROCESSING_BRANCH, SARTHAK_ROOT)
print("Reproducibility: benchmarking pinned historical Role 3 revision "
      f"{EXPECTED_ROLE3_COMMIT}, not the latest {ROLE3_BRANCH} branch HEAD.")
try:
    subprocess.run(
        ["git", "-C", str(ROLE3_ROOT), "checkout", "--detach", EXPECTED_ROLE3_COMMIT],
        check=True, text=True, capture_output=True,
    )
except subprocess.CalledProcessError as exc:
    details = (exc.stderr or exc.stdout or str(exc)).strip()
    raise RuntimeError(
        f"Could not check out pinned Role 3 commit {EXPECTED_ROLE3_COMMIT} in detached HEAD mode. "
        f"Confirm the commit is available in the cloned repository. Git said: {details}"
    ) from exc

def git_head(root):
    return subprocess.run(
        ["git", "-C", str(root), "rev-parse", "HEAD"],
        check=True, text=True, capture_output=True,
    ).stdout.strip()

ROLE3_COMMIT = git_head(ROLE3_ROOT)
PREPROCESSING_COMMIT = git_head(SARTHAK_ROOT)
if ROLE3_COMMIT != EXPECTED_ROLE3_COMMIT:
    raise RuntimeError(
        f"Expected audited Role 3 commit {EXPECTED_ROLE3_COMMIT}, got {ROLE3_COMMIT}"
    )
if PREPROCESSING_COMMIT != EXPECTED_PREPROCESSING_COMMIT:
    raise RuntimeError(
        f"Expected audited Sarthak commit {EXPECTED_PREPROCESSING_COMMIT}, got {PREPROCESSING_COMMIT}"
    )
required = [
    ROLE3_ROOT / "advanced_models/vision/feature_extractor.py",
    ROLE3_ROOT / "advanced_models/temporal/temporal_encoder.py",
    ROLE3_ROOT / "advanced_models/video_temporal_pipeline.py",
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("Role 3 files missing from branch: " + ", ".join(missing))
print("Role 3:", ROLE3_BRANCH, ROLE3_COMMIT)
print("Preprocessing:", PREPROCESSING_BRANCH, PREPROCESSING_COMMIT)

In [ ]:
import sys

# Keep checkouts separate: advanced_models from Role 3, preprocessing from Sarthak.
sys.path.insert(0, str(SARTHAK_ROOT))
sys.path.insert(0, str(ROLE3_ROOT))

import cv2
import numpy as np
import transformers
from preprocessing import ReVADataset, collate_reva, read_annotations
from preprocessing.hf_access import fetch_annotation_file
from advanced_models.temporal.temporal_encoder import TransformerTemporalEncoder
from advanced_models.video_temporal_pipeline import VideoTemporalPipeline
from advanced_models.vision.feature_extractor import CLIP_MODEL_ID, CLIPVisionFeatureExtractor
from transformers import CLIPVisionModelWithProjection

print("Transformers:", transformers.__version__)
print("OpenCV:", cv2.__version__)
print("NumPy:", np.__version__)
print("Preprocessing source:", Path(sys.modules["preprocessing"].__file__).resolve())
print("Role 3 source:", Path(sys.modules["advanced_models.video_temporal_pipeline"].__file__).resolve())

## Load actual validation annotations and select a subset

The audited manifest pins ReVA revision `06224fa1b42e5e35677a8b7f2145e06307a9a4ee` and identifies `val.json`. Selection uses the observed `qa_id`, `video_path`, and `category` fields. It round-robins over categories while preferring one sample per distinct video. Any fallback that reuses a video is visible in the reported distinct-video count.

In [ ]:
import json
import random
from collections import Counter, defaultdict

DATASET_REPO_ID = "ReVA-Benchmark/ReVA"
DATASET_REVISION_EXPECTED = "06224fa1b42e5e35677a8b7f2145e06307a9a4ee"
FRAME_COUNTS = (4, 8, 16)
TARGET_SAMPLE_COUNT = 32
SELECTION_SEED = 2026
CACHE_ROOT = Path("/content/reva_role3_subset_cache")
HF_CACHE_DIR = CACHE_ROOT / "huggingface"
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)

with (SARTHAK_ROOT / "manifests/reva_manifest.json").open(encoding="utf-8") as f:
    manifest = json.load(f)
DATASET_REVISION = manifest["revision"]
if DATASET_REVISION != DATASET_REVISION_EXPECTED:
    raise RuntimeError(f"Unexpected ReVA dataset revision: {DATASET_REVISION}")
if "val.json" not in manifest["files"]:
    raise FileNotFoundError("Pinned manifest does not list actual validation annotation val.json")

ANNOTATION_PATH = fetch_annotation_file(
    DATASET_REPO_ID, "val.json", revision=DATASET_REVISION, cache_dir=HF_CACHE_DIR
)
VALIDATION_ROWS = read_annotations(ANNOTATION_PATH)
if not VALIDATION_ROWS:
    raise ValueError(f"No validation records found in {ANNOTATION_PATH}")

def row_id(row):
    for key in ("qa_id", "sample_id", "id"):
        value = row.get(key)
        if value is not None and str(value).strip():
            return str(value)
    return None

usable_rows, excluded_rows, seen_ids = [], [], set()
for row in VALIDATION_ROWS:
    sid = row_id(row)
    video = row.get("video_path", row.get("file_path"))
    if sid is None or not video:
        excluded_rows.append({"sample_id": sid, "reason": "missing actual sample ID or video path"})
        continue
    if sid in seen_ids:
        excluded_rows.append({"sample_id": sid, "reason": "duplicate sample ID"})
        continue
    seen_ids.add(sid)
    item = dict(row)
    item["_sample_id"] = sid
    item["_video_path"] = str(video)
    item["_category"] = str(row.get("category") or "Unknown")
    usable_rows.append(item)

rng = random.Random(SELECTION_SEED)
by_category = defaultdict(list)
for row in usable_rows:
    by_category[row["_category"]].append(row)
for rows in by_category.values():
    rng.shuffle(rows)

selected, chosen_ids, chosen_videos = [], set(), set()
categories = sorted(by_category)
target = min(TARGET_SAMPLE_COUNT, len(usable_rows))
while len(selected) < target:
    progressed = False
    for category in categories:
        candidate = next((row for row in by_category[category]
                          if row["_sample_id"] not in chosen_ids
                          and row["_video_path"] not in chosen_videos), None)
        if candidate is not None:
            selected.append(candidate)
            chosen_ids.add(candidate["_sample_id"])
            chosen_videos.add(candidate["_video_path"])
            progressed = True
            if len(selected) >= target:
                break
    if not progressed:
        break

# If the split has too few distinct videos, use additional real QA records.
if len(selected) < target:
    rest = [row for row in usable_rows if row["_sample_id"] not in chosen_ids]
    rng.shuffle(rest)
    selected.extend(rest[:target - len(selected)])

SAMPLE_IDS = [row["_sample_id"] for row in selected]
SELECTED_BY_ID = {row["_sample_id"]: row for row in selected}
SELECTED_CATEGORY_COUNTS = dict(sorted(Counter(row["_category"] for row in selected).items()))
SELECTED_DISTINCT_VIDEOS = len({row["_video_path"] for row in selected})
print("Annotation:", ANNOTATION_PATH)
print("Dataset revision:", DATASET_REVISION)
print("Validation records:", len(VALIDATION_ROWS))
print("Selected records:", len(SAMPLE_IDS))
print("Distinct videos selected:", SELECTED_DISTINCT_VIDEOS)
print("Category distribution:", SELECTED_CATEGORY_COUNTS)
print("Excluded malformed/duplicate rows:", len(excluded_rows))

## Load CLIP and the committed Role 3 pipeline

In [ ]:
MODEL_LOAD_ERROR = None
MODEL_REVISION = None
PIPELINE = None
try:
    vision_model = CLIPVisionModelWithProjection.from_pretrained(CLIP_MODEL_ID)
    MODEL_REVISION = getattr(vision_model.config, "_commit_hash", None)
    extractor = CLIPVisionFeatureExtractor(
        vision_model,
        input_normalization="imagenet",
        frame_chunk_size=4,
        cuda_mixed_precision=torch.cuda.is_available(),
        output_dtype=torch.float32,
    )
    temporal = TransformerTemporalEncoder(
        embedding_dim=512, num_heads=8, num_layers=2, dropout=0.1, max_seq_len=16
    )
    PIPELINE = VideoTemporalPipeline(extractor, temporal).to(DEVICE)
    PIPELINE.eval()
    print("Checkpoint:", CLIP_MODEL_ID, "revision:", MODEL_REVISION, "device:", DEVICE)
except Exception as exc:
    MODEL_LOAD_ERROR = {"type": type(exc).__name__, "message": str(exc)}
    print(f"MODEL LOAD ERROR ({type(exc).__name__}): {exc}")
    print("Resolution: verify Colab internet access and RAM/disk/GPU memory. No fallback checkpoint is used.")

## Process real samples at each frame count

This uses Sarthak's actual `ReVADataset` constructor and `collate_reva`. `missing_video="raise"` makes a missing/corrupt video an explicit per-sample failure. Video loading and frame preprocessing are not timed as model inference.

In [ ]:
SAMPLE_RESULTS = {
    sid: {
        "sample_id": sid,
        "video_path": SELECTED_BY_ID[sid]["_video_path"],
        "category": SELECTED_BY_ID[sid]["_category"],
        "task": SELECTED_BY_ID[sid].get("task"),
        "frame_counts": {},
    }
    for sid in SAMPLE_IDS
}
FRAME_RUN_ERRORS = []

def make_dataset(num_frames, sample_ids):
    return ReVADataset(
        annotation_file=ANNOTATION_PATH,
        video_root=None,
        num_frames=num_frames,
        sampling="uniform",
        split="val",
        sample_ids=sample_ids,
        image_size=(224, 224),
        normalize=True,
        cache_dir=None,
        missing_video="raise",
        dataset_repo_id=DATASET_REPO_ID,
        revision=DATASET_REVISION,
        hf_cache_dir=HF_CACHE_DIR,
        local_files_only=False,
    )

def forward_one_with_oom_retry(batch):
    inputs = batch["video_frames"].to(DEVICE)
    while True:
        chunk = PIPELINE.feature_extractor.frame_chunk_size
        try:
            with torch.inference_mode():
                outputs = PIPELINE({"video_frames": inputs})
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            return outputs, inputs, chunk
        except torch.cuda.OutOfMemoryError:
            if DEVICE.type != "cuda" or chunk is None or chunk <= 1:
                raise
            next_chunk = max(1, chunk // 2)
            print(f"CUDA OOM at chunk {chunk}; retrying this real input with chunk {next_chunk}")
            PIPELINE.feature_extractor.frame_chunk_size = next_chunk
            torch.cuda.empty_cache()

if PIPELINE is None:
    for sid in SAMPLE_IDS:
        for count in FRAME_COUNTS:
            SAMPLE_RESULTS[sid]["frame_counts"][str(count)] = {
                "status": "not_executed", "error": MODEL_LOAD_ERROR
            }
else:
    PIPELINE.eval()
    for frame_count in FRAME_COUNTS:
        try:
            dataset = make_dataset(frame_count, SAMPLE_IDS)
        except Exception as exc:
            error = {"type": type(exc).__name__, "message": str(exc)}
            for sid in SAMPLE_IDS:
                SAMPLE_RESULTS[sid]["frame_counts"][str(frame_count)] = {
                    "status": "failure", "error": error
                }
            FRAME_RUN_ERRORS.append({"frame_count": frame_count, "error": error})
            continue

        found_ids = set()
        for index, row in enumerate(dataset.rows):
            sid = row_id(row)
            if sid is None or sid not in SAMPLE_RESULTS:
                continue
            found_ids.add(sid)
            record = SAMPLE_RESULTS[sid]
            try:
                sample = dataset[index]  # errors raise; no zero-frame fallback
                batch = collate_reva([sample])
                if not torch.is_tensor(batch.get("video_frames")):
                    raise TypeError("collate_reva did not return tensor video_frames")
                if "video_error" in batch:
                    errors = batch["video_error"]
                    if not isinstance(errors, (list, tuple)) or any(
                        value is not None and str(value).strip() for value in errors
                    ):
                        raise ValueError(f"video_error metadata present: {errors}")
                actual_sid = str((batch.get("sample_id") or [sid])[0])
                if actual_sid != sid:
                    raise ValueError(f"Collated ID mismatch: expected {sid}, got {actual_sid}")

                frames = batch["video_frames"]
                expected_shape = (1, frame_count, 3, 224, 224)
                if tuple(frames.shape) != expected_shape:
                    raise ValueError(f"Expected {expected_shape}, got {tuple(frames.shape)}")
                if frames.dtype != torch.float32:
                    raise TypeError(f"Expected float32 input, got {frames.dtype}")
                if not torch.isfinite(frames).all():
                    raise ValueError("Input frames contain NaN or Inf")

                outputs, inputs_device, chunk = forward_one_with_oom_retry(batch)
                frame_features = outputs["frame_features"]
                video_features = outputs["video_features"]
                has_nan = bool(torch.isnan(frame_features).any().item() or torch.isnan(video_features).any().item())
                has_inf = bool(torch.isinf(frame_features).any().item() or torch.isinf(video_features).any().item())
                if has_nan or has_inf:
                    raise ValueError(f"Non-finite features (NaN={has_nan}, Inf={has_inf})")
                record["frame_counts"][str(frame_count)] = {
                    "status": "success",
                    "input_shape": list(frames.shape),
                    "frame_features_shape": list(frame_features.shape),
                    "video_features_shape": list(video_features.shape),
                    "output_dtype": str(video_features.dtype),
                    "output_device": str(video_features.device),
                    "contains_nan": has_nan,
                    "contains_inf": has_inf,
                    "frame_chunk_size_used": chunk,
                }
                del outputs, inputs_device, frame_features, video_features, frames, batch, sample
            except Exception as exc:
                error = {"type": type(exc).__name__, "message": str(exc)}
                record["frame_counts"][str(frame_count)] = {
                    "status": "failure",
                    "error": error,
                    "frame_chunk_size_used": PIPELINE.feature_extractor.frame_chunk_size,
                }
                FRAME_RUN_ERRORS.append({
                    "sample_id": sid, "video_path": record["video_path"],
                    "frame_count": frame_count, "error": error,
                })
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

        for sid in set(SAMPLE_IDS) - found_ids:
            error = {"type": "SampleNotFound", "message": "ID absent from ReVADataset rows for pinned val.json"}
            SAMPLE_RESULTS[sid]["frame_counts"][str(frame_count)] = {"status": "failure", "error": error}
            FRAME_RUN_ERRORS.append({"sample_id": sid, "frame_count": frame_count, "error": error})

print("Selected:", len(SAMPLE_IDS), "per-sample/frame-count attempts:", len(SAMPLE_IDS) * len(FRAME_COUNTS))
print("Recorded per-sample failures:", len(FRAME_RUN_ERRORS))

## Benchmark helper and methodology

The helper below is isolated in its own cell so it can be tested offline. It executes 3 warm-ups and 10 measured runs by default, uses inference mode, synchronizes before and after each CUDA pass, and reports mean, median, and sample standard deviation. Peak allocated CUDA memory is measured after warm-up and a peak-stat reset.

In [ ]:
import statistics
import time

def benchmark_callable(forward, *, device, warmup_runs=3, measured_runs=10):
    """Measure callable forward latency with inference mode and synchronized CUDA timing."""
    device = torch.device(device)
    if not callable(forward):
        raise TypeError("forward must be callable")
    if warmup_runs < 0:
        raise ValueError("warmup_runs must be non-negative")
    if measured_runs < 2:
        raise ValueError("measured_runs must be at least 2 for standard deviation")
    if device.type == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA benchmark requested but CUDA is unavailable")

    def synchronize():
        if device.type == "cuda":
            torch.cuda.synchronize(device)

    with torch.inference_mode():
        for _ in range(warmup_runs):
            synchronize()
            output = forward()
            synchronize()
            del output
        if device.type == "cuda":
            synchronize()
            torch.cuda.reset_peak_memory_stats(device)

        latencies = []
        for _ in range(measured_runs):
            synchronize()
            start = time.perf_counter()
            output = forward()
            synchronize()
            latencies.append(time.perf_counter() - start)
            del output

    peak_memory_mb = (
        torch.cuda.max_memory_allocated(device) / 1024**2
        if device.type == "cuda" else None
    )
    return {
        "warmup_runs": warmup_runs,
        "measured_runs": measured_runs,
        "latencies_seconds": latencies,
        "mean_seconds": statistics.mean(latencies),
        "median_seconds": statistics.median(latencies),
        "standard_deviation_seconds": statistics.stdev(latencies),
        "standard_deviation_kind": "sample",
        "peak_gpu_memory_mb": peak_memory_mb,
    }

In [ ]:
BENCHMARK_SAMPLE_ID = None
BENCHMARK_RESULTS = []
BENCHMARK_ERRORS = []
COMMON_CHUNK_SIZE = None

common_ids = [
    sid for sid in SAMPLE_IDS
    if all(SAMPLE_RESULTS[sid]["frame_counts"].get(str(count), {}).get("status") == "success"
           for count in FRAME_COUNTS)
]
if common_ids and PIPELINE is not None:
    BENCHMARK_SAMPLE_ID = common_ids[0]
    # Untimed T=16 probe chooses one chunk size that also applies to T=4 and T=8.
    probe_dataset = make_dataset(16, [BENCHMARK_SAMPLE_ID])
    probe_index = next(i for i, row in enumerate(probe_dataset.rows) if row_id(row) == BENCHMARK_SAMPLE_ID)
    probe = collate_reva([probe_dataset[probe_index]])
    candidate = 4
    while True:
        PIPELINE.feature_extractor.frame_chunk_size = candidate
        try:
            with torch.inference_mode():
                probe_output = PIPELINE({"video_frames": probe["video_frames"].to(DEVICE)})
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            del probe_output
            COMMON_CHUNK_SIZE = candidate
            break
        except torch.cuda.OutOfMemoryError as exc:
            if DEVICE.type != "cuda" or candidate <= 1:
                BENCHMARK_ERRORS.append({"stage": "T=16 chunk probe", "type": type(exc).__name__, "message": str(exc)})
                break
            candidate = max(1, candidate // 2)
            torch.cuda.empty_cache()
    del probe
else:
    BENCHMARK_ERRORS.append({
        "stage": "sample selection",
        "type": "NoCommonBenchmarkSample",
        "message": "No selected validation sample succeeded at T=4, 8, and 16.",
    })

if COMMON_CHUNK_SIZE is not None:
    PIPELINE.eval()
    for count in FRAME_COUNTS:
        try:
            ds = make_dataset(count, [BENCHMARK_SAMPLE_ID])
            i = next(i for i, row in enumerate(ds.rows) if row_id(row) == BENCHMARK_SAMPLE_ID)
            benchmark_batch = collate_reva([ds[i]])
            benchmark_inputs = benchmark_batch["video_frames"].to(DEVICE)
            PIPELINE.feature_extractor.frame_chunk_size = COMMON_CHUNK_SIZE
            metrics = benchmark_callable(
                lambda: PIPELINE({"video_frames": benchmark_inputs}),
                device=DEVICE, warmup_runs=3, measured_runs=10,
            )
            metrics.update({
                "status": "success", "frame_count": count,
                "sample_id": BENCHMARK_SAMPLE_ID, "batch_size": 1,
                "input_shape": list(benchmark_inputs.shape),
                "frame_features_shape": [1, count, 512],
                "video_features_shape": [1, 512],
                "frame_chunk_size": COMMON_CHUNK_SIZE,
            })
            BENCHMARK_RESULTS.append(metrics)
            print(
                f"T={count} sample={BENCHMARK_SAMPLE_ID}: mean={metrics['mean_seconds']:.6f}s, "
                f"median={metrics['median_seconds']:.6f}s, sample_std={metrics['standard_deviation_seconds']:.6f}s, "
                f"peak_GPU_MiB={metrics['peak_gpu_memory_mb']}, chunk={COMMON_CHUNK_SIZE}"
            )
            del benchmark_inputs, benchmark_batch
        except Exception as exc:
            error = {"type": type(exc).__name__, "message": str(exc)}
            BENCHMARK_ERRORS.append({"frame_count": count, **error})
            BENCHMARK_RESULTS.append({
                "status": "failure", "frame_count": count,
                "sample_id": BENCHMARK_SAMPLE_ID, "batch_size": 1,
                "warmup_runs": 3, "measured_runs": 10,
                "mean_seconds": None, "median_seconds": None,
                "standard_deviation_seconds": None, "peak_gpu_memory_mb": None,
                "frame_chunk_size": COMMON_CHUNK_SIZE, "error": error,
            })
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
else:
    for count in FRAME_COUNTS:
        BENCHMARK_RESULTS.append({
            "status": "not_executed", "frame_count": count,
            "sample_id": None, "batch_size": 1,
            "warmup_runs": 3, "measured_runs": 10,
            "mean_seconds": None, "median_seconds": None,
            "standard_deviation_seconds": None, "peak_gpu_memory_mb": None,
            "frame_chunk_size": None,
        })

In [ ]:
from IPython.display import Markdown, display

headers = ["T", "Sample ID", "Mean s", "Median s", "Sample std s", "Peak GPU MiB", "Chunk", "Status"]
table_rows = []
for item in BENCHMARK_RESULTS:
    def show(value):
        return "—" if value is None else (f"{value:.6f}" if isinstance(value, float) else str(value))
    table_rows.append([
        item["frame_count"], item.get("sample_id"), show(item.get("mean_seconds")),
        show(item.get("median_seconds")), show(item.get("standard_deviation_seconds")),
        show(item.get("peak_gpu_memory_mb")), show(item.get("frame_chunk_size")), item["status"],
    ])
lines = ["| " + " | ".join(headers) + " |", "|" + "|".join(["---"] * len(headers)) + "|"]
lines.extend("| " + " | ".join(map(str, row)) + " |" for row in table_rows)
display(Markdown("\n".join(lines)))

successful_all_t = [
    sid for sid in SAMPLE_IDS
    if all(SAMPLE_RESULTS[sid]["frame_counts"].get(str(count), {}).get("status") == "success"
           for count in FRAME_COUNTS)
]
print("Distinct selected videos:", SELECTED_DISTINCT_VIDEOS)
print("Samples successful at all frame counts:", len(successful_all_t))
print("Category counts selected:", SELECTED_CATEGORY_COUNTS)
print("Category counts successful at all T:", dict(sorted(Counter(SAMPLE_RESULTS[sid]["category"] for sid in successful_all_t).items())))

## Save reproducibility results

The report is written to a new file in the Role 3 clone. It records actual measurements/errors; no QA metrics are computed. The existing `role3_real_inference_results.json` is left untouched.

In [ ]:
from datetime import datetime, timezone

RESULTS_PATH = ROLE3_ROOT / "reports/role3_validation_subset_benchmark.json"
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
all_statuses = [
    SAMPLE_RESULTS[sid]["frame_counts"].get(str(count), {}).get("status")
    for sid in SAMPLE_IDS for count in FRAME_COUNTS
]
all_sample_success = bool(all_statuses) and all(status == "success" for status in all_statuses)
all_benchmark_success = bool(BENCHMARK_RESULTS) and all(item["status"] == "success" for item in BENCHMARK_RESULTS)
if all_sample_success and all_benchmark_success:
    overall_status = "completed"
elif any(status == "success" for status in all_statuses) or any(item["status"] == "success" for item in BENCHMARK_RESULTS):
    overall_status = "partial"
else:
    overall_status = "failed"

successful_all_t = [
    sid for sid in SAMPLE_IDS
    if all(SAMPLE_RESULTS[sid]["frame_counts"].get(str(count), {}).get("status") == "success"
           for count in FRAME_COUNTS)
]
report = {
    "schema_version": "1.0",
    "status": overall_status,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "scope": "Visual-temporal feature extraction/benchmark only; no QA accuracy or temporal reasoning improvement measured.",
    "repository": {
        "role3_branch": ROLE3_BRANCH, "role3_commit": ROLE3_COMMIT,
        "preprocessing_branch": PREPROCESSING_BRANCH, "preprocessing_commit": PREPROCESSING_COMMIT,
    },
    "dataset": {
        "name": "ReVA", "repo_id": DATASET_REPO_ID, "revision": DATASET_REVISION,
        "validation_annotation_file": "val.json",
        "validation_records_parsed": len(VALIDATION_ROWS),
        "samples_attempted": len(SAMPLE_IDS),
        "samples_successful_at_all_frame_counts": len(successful_all_t),
        "distinct_videos_selected": SELECTED_DISTINCT_VIDEOS,
        "distinct_videos_successful_at_all_frame_counts": len({SAMPLE_RESULTS[sid]["video_path"] for sid in successful_all_t}),
        "selection_seed": SELECTION_SEED,
        "category_distribution_selected": SELECTED_CATEGORY_COUNTS,
        "category_distribution_successful_at_all_frame_counts": dict(sorted(Counter(SAMPLE_RESULTS[sid]["category"] for sid in successful_all_t).items())),
    },
    "environment": {
        "python_version": platform.python_version(),
        "pytorch_version": torch.__version__,
        "transformers_version": transformers.__version__,
        "device": str(DEVICE), "gpu_name": GPU_NAME,
        "gpu_total_memory_mb": GPU_TOTAL_MEMORY_MB,
        "gpu_free_memory_at_start_mb": GPU_FREE_MEMORY_MB,
    },
    "model": {
        "checkpoint_id": CLIP_MODEL_ID, "checkpoint_revision": MODEL_REVISION,
        "class": "CLIPVisionModelWithProjection", "projected_dimension": 512,
        "visual_parameters_frozen": bool(PIPELINE is not None and all(
            not p.requires_grad for p in PIPELINE.feature_extractor.vision_model.parameters()
        )),
    },
    "preprocessing": {
        "frame_counts": list(FRAME_COUNTS), "batch_size": 1,
        "sampling": "uniform", "image_size": [224, 224],
        "channel_order": "RGB", "input_dtype": "torch.float32",
        "normalization": "ImageNet input; Role 3 extractor converts to CLIP normalization",
        "missing_video": "raise", "feature_cache": False,
    },
    "benchmark_method": {
        "sample_id": BENCHMARK_SAMPLE_ID, "same_sample_for_all_frame_counts": BENCHMARK_SAMPLE_ID is not None,
        "batch_size": 1, "warmup_runs": 3, "measured_runs": 10,
        "statistics": "mean, median, sample standard deviation",
        "synchronize_before_and_after_each_cuda_timed_run": True,
        "timing_scope": "pipeline forward; excludes download/decode/sampling/resize/ImageNet preprocessing/host-to-device transfer; includes CLIP normalization and both model forwards",
        "peak_memory_scope": "max allocated CUDA memory after warm-up and reset; model and benchmark input are resident",
        "common_frame_chunk_size": COMMON_CHUNK_SIZE,
    },
    "benchmarks": BENCHMARK_RESULTS,
    "per_sample_results": [SAMPLE_RESULTS[sid] for sid in SAMPLE_IDS],
    "errors_and_skipped_samples": FRAME_RUN_ERRORS + BENCHMARK_ERRORS,
    "excluded_annotation_rows": excluded_rows,
}
with RESULTS_PATH.open("w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
    f.write("\n")
print("Saved:", RESULTS_PATH)
print("Overall status:", overall_status)
print("Existing single-sample JSON was not modified:", ROLE3_ROOT / "reports/role3_real_inference_results.json")

DOWNLOAD_RESULTS_TO_LOCAL = False
if DOWNLOAD_RESULTS_TO_LOCAL:
    from google.colab import files
    files.download(str(RESULTS_PATH))